# Amazon Review Data 2018 — Week 1 Preprocessing V4

This version replaces category-based positive labels with **Amazon product relationship signals** from `also_buy` and `also_view` / `also_viewed`.

### V4 label strategy
- **3** = source product
- **2** = direct Amazon relation (`also_buy` / `also_view`)
- **1** = second-hop graph relation or same-brand fallback
- **0** = lexical hard negative + random negative

TF-IDF is used **only to mine hard negatives**, not to generate positive labels. This reduces circular evaluation when you later test BM25 / TF-IDF baselines.

**Run the single code cell below in Google Colab.**


In [ ]:

# ============================================================
# Amazon Review Data 2018 — Week 1 Preprocessing Pipeline V4
# Relation-Graph Labels (also_buy / also_view) + Hard Negatives
#
# Goal:
#   Build a stronger Week-1 ranking dataset WITHOUT relying on
#   category hierarchy for positive labels.
#
# Core label logic:
#   3 = source product for the pseudo query
#   2 = DIRECT Amazon relation (also_buy / also_view)
#   1 = WEAK graph relation (2-hop relation) or same-brand fallback
#   0 = unrelated hard/random negatives
#
# IMPORTANT:
# Amazon 2018 does NOT contain real search queries or explicit
# search relevance judgments. These are still pseudo / heuristic
# labels, but V4 uses Amazon-provided product relationship signals
# instead of using TF-IDF to create positive labels.
# ============================================================

!pip -q install pandas pyarrow scikit-learn tqdm requests

import os
import re
import ast
import random
import shutil
import warnings
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import requests
from tqdm.auto import tqdm
from IPython.display import display, Markdown

from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.metrics.pairwise import linear_kernel

warnings.filterwarnings("ignore")
random.seed(42)
np.random.seed(42)

# ============================================================
# 1. CONFIG
# ============================================================

DATASET_CATEGORY = "All_Beauty"

# Final development catalog size.
N_PRODUCTS = 10000

# Number of pseudo search queries.
N_QUERIES = 1000

# Candidate composition per query.
N_LABEL2 = 2        # direct Amazon relation
N_LABEL1 = 2        # second-hop / same-brand weak relation
N_HARD_NEG = 2      # lexically confusing negatives
N_RANDOM_NEG = 1    # random negative

RANDOM_STATE = 42

BASE_DIR = Path("/content/amazon_search_project_v4")
RAW_DIR = BASE_DIR / "data" / "raw"
OUT_DIR = BASE_DIR / "data" / "processed"

RAW_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)

REVIEWS_URL = (
    "https://mcauleylab.ucsd.edu/public_datasets/data/amazon_v2/"
    f"categoryFiles/{DATASET_CATEGORY}.json.gz"
)

META_URL = (
    "https://mcauleylab.ucsd.edu/public_datasets/data/amazon_v2/"
    f"metaFiles2/meta_{DATASET_CATEGORY}.json.gz"
)

REVIEWS_PATH = RAW_DIR / f"{DATASET_CATEGORY}.json.gz"
META_PATH = RAW_DIR / f"meta_{DATASET_CATEGORY}.json.gz"


# ============================================================
# 2. DOWNLOAD
# ============================================================

def download_file(url, path):
    if path.exists():
        print("✓ Already exists:", path.name)
        return

    print("Downloading:", path.name)

    r = requests.get(url, stream=True, timeout=120)
    r.raise_for_status()

    total = int(r.headers.get("content-length", 0))

    with open(path, "wb") as f, tqdm(
        total=total,
        unit="B",
        unit_scale=True,
        desc=path.name
    ) as pbar:
        for chunk in r.iter_content(chunk_size=1024 * 1024):
            if chunk:
                f.write(chunk)
                pbar.update(len(chunk))

download_file(META_URL, META_PATH)
download_file(REVIEWS_URL, REVIEWS_PATH)


# ============================================================
# 3. LOAD RAW DATA
# ============================================================

print("\nLoading Amazon 2018 metadata...")
meta_raw = pd.read_json(
    META_PATH,
    lines=True,
    compression="gzip"
)

print("Metadata shape:", meta_raw.shape)

print("\nLoading Amazon 2018 reviews...")
reviews_raw = pd.read_json(
    REVIEWS_PATH,
    lines=True,
    compression="gzip"
)

print("Reviews shape:", reviews_raw.shape)

print("\nRaw metadata columns:")
print(list(meta_raw.columns))

print("\nRelation fields present:")
for c in ["also_buy", "also_view", "also_viewed"]:
    print(f"{c}: {c in meta_raw.columns}")


# ============================================================
# 4. HELPERS
# ============================================================

def is_missing(x):
    return x is None or (isinstance(x, float) and np.isnan(x))


def list_to_text(x):
    if is_missing(x):
        return ""

    if isinstance(x, list):
        parts = []
        for item in x:
            if isinstance(item, list):
                parts.extend(str(v) for v in item if not is_missing(v))
            elif isinstance(item, dict):
                parts.extend(f"{k}: {v}" for k, v in item.items())
            elif not is_missing(item):
                parts.append(str(item))
        return " ".join(parts)

    if isinstance(x, dict):
        return " ".join(f"{k}: {v}" for k, v in x.items())

    return str(x)


def clean_text(x):
    x = list_to_text(x)
    x = x.replace("\n", " ").replace("\r", " ").replace("\t", " ")
    x = re.sub(r"<[^>]+>", " ", x)
    x = re.sub(r"\s+", " ", x).strip()
    return x


def normalize_query(q):
    q = clean_text(q).lower()
    q = re.sub(r"[^a-z0-9\s]", " ", q)
    q = re.sub(r"\s+", " ", q).strip()
    return q


def parse_price(x):
    if is_missing(x):
        return np.nan

    if isinstance(x, (int, float, np.number)):
        return float(x)

    nums = re.findall(r"\d+(?:\.\d+)?", str(x))
    return float(nums[0]) if nums else np.nan


def get_first_image(row):
    for col in ["imageURLHighRes", "imageURL"]:
        if col not in row.index:
            continue

        value = row[col]

        if isinstance(value, list) and len(value) > 0:
            return str(value[0]).strip()

        if isinstance(value, str) and value.strip():
            return value.strip()

    return ""


def normalize_asin_list(x):
    """
    Convert relation field into a clean list of ASIN strings.
    Handles list / tuple / stringified list / missing.
    """
    if is_missing(x):
        return []

    if isinstance(x, (list, tuple, set)):
        vals = list(x)

    elif isinstance(x, str):
        s = x.strip()

        if not s:
            return []

        # Sometimes connectors/parsers may stringify Python lists.
        if s.startswith("[") and s.endswith("]"):
            try:
                parsed = ast.literal_eval(s)
                vals = parsed if isinstance(parsed, list) else [parsed]
            except:
                vals = [s]
        else:
            vals = [s]

    else:
        vals = [x]

    out = []

    for v in vals:
        if is_missing(v):
            continue

        asin = str(v).strip()

        if asin and asin.lower() not in {"nan", "none"}:
            out.append(asin)

    # dedupe while keeping order
    return list(dict.fromkeys(out))


def normalize_category_value(x):
    """
    Best-effort category parsing only for analysis.
    V4 does NOT rely on category for relevance labels.
    """
    if is_missing(x):
        return "unknown"

    if isinstance(x, list):
        flat = []

        for item in x:
            if isinstance(item, list):
                flat.extend(clean_text(v) for v in item if clean_text(v))
            else:
                value = clean_text(item)
                if value:
                    flat.append(value)

        return flat[-1] if flat else "unknown"

    value = clean_text(x)
    return value if value else "unknown"


# ============================================================
# 5. CLEAN FULL PRODUCT CATALOG
# ============================================================

meta = meta_raw.copy()

for col in [
    "asin", "title", "description", "feature", "brand", "price",
    "imageURL", "imageURLHighRes", "also_buy", "also_view", "also_viewed"
]:
    if col not in meta.columns:
        meta[col] = np.nan

# category/categoryies are optional here
if "category" in meta.columns:
    raw_category = meta["category"]
elif "categories" in meta.columns:
    raw_category = meta["categories"]
else:
    raw_category = pd.Series([None] * len(meta), index=meta.index)

products_full = pd.DataFrame({
    "product_id": meta["asin"].astype(str),
    "asin": meta["asin"].astype(str),
    "title": meta["title"].apply(clean_text),
    "description": meta["description"].apply(clean_text),
    "features": meta["feature"].apply(clean_text),
    "brand": meta["brand"].apply(clean_text),
    "price": meta["price"].apply(parse_price),
    "image_url": meta.apply(get_first_image, axis=1),
    "category": raw_category.apply(normalize_category_value),
    "also_buy": meta["also_buy"].apply(normalize_asin_list),
    # Support both possible field names.
    "also_view": meta.apply(
        lambda r: normalize_asin_list(r["also_view"])
        if "also_view" in meta.columns and isinstance(r["also_view"], (list, tuple, set, str))
        else normalize_asin_list(r["also_viewed"])
        if "also_viewed" in meta.columns
        else [],
        axis=1
    )
})

# Basic product cleaning
products_full["title"] = products_full["title"].replace({"": np.nan})

products_full = products_full.dropna(
    subset=["product_id", "title"]
)

products_full = products_full[
    (products_full["product_id"] != "nan")
    &
    (products_full["product_id"].str.len() > 0)
].copy()

products_full = products_full.drop_duplicates(
    "product_id",
    keep="first"
).reset_index(drop=True)

# Product-id universe
full_ids = set(products_full["product_id"])

# Keep only relation links that are present in this All_Beauty metadata.
products_full["also_buy_in_catalog"] = products_full["also_buy"].apply(
    lambda xs: [x for x in xs if x in full_ids]
)

products_full["also_view_in_catalog"] = products_full["also_view"].apply(
    lambda xs: [x for x in xs if x in full_ids]
)

products_full["direct_related"] = products_full.apply(
    lambda r: list(dict.fromkeys(
        r["also_buy_in_catalog"] + r["also_view_in_catalog"]
    )),
    axis=1
)

products_full["direct_relation_count"] = products_full[
    "direct_related"
].apply(len)

print("\nFull cleaned product catalog:", products_full.shape)

relation_summary = pd.DataFrame({
    "metric": [
        "products_with_also_buy",
        "products_with_also_view",
        "products_with_any_direct_relation",
        "mean_direct_relations",
        "median_direct_relations"
    ],
    "value": [
        int(products_full["also_buy_in_catalog"].apply(len).gt(0).sum()),
        int(products_full["also_view_in_catalog"].apply(len).gt(0).sum()),
        int(products_full["direct_relation_count"].gt(0).sum()),
        float(products_full["direct_relation_count"].mean()),
        float(products_full["direct_relation_count"].median())
    ]
})

display(Markdown("## Amazon Relation Graph Diagnostics"))
display(relation_summary)


# ============================================================
# 6. REVIEW AGGREGATION
# ============================================================

reviews = reviews_raw.copy()
reviews["asin"] = reviews["asin"].astype(str)

if "overall" not in reviews.columns:
    reviews["overall"] = np.nan

review_stats = (
    reviews.groupby("asin")
    .agg(
        average_rating=("overall", "mean"),
        review_count=("overall", "size")
    )
    .reset_index()
    .rename(columns={"asin": "product_id"})
)

products_full = products_full.merge(
    review_stats,
    on="product_id",
    how="left"
)

products_full["average_rating"] = products_full["average_rating"].fillna(0.0)
products_full["review_count"] = products_full["review_count"].fillna(0).astype(int)

# Missing indicators
products_full["price_missing"] = products_full["price"].isna().astype(int)
products_full["description_missing"] = products_full["description"].eq("").astype(int)
products_full["image_missing"] = products_full["image_url"].eq("").astype(int)
products_full["brand_missing"] = products_full["brand"].eq("").astype(int)

# Search text
products_full["product_text"] = (
    products_full["title"].fillna("") + " " +
    products_full["brand"].fillna("") + " " +
    products_full["features"].fillna("") + " " +
    products_full["description"].fillna("")
).str.replace(r"\s+", " ", regex=True).str.strip()


# ============================================================
# 7. STRICT PSEUDO QUERY GENERATION
# ============================================================

STOPWORDS = set(ENGLISH_STOP_WORDS)

NOISE = {
    "great", "good", "nice", "love", "loved", "excellent", "perfect",
    "awesome", "amazing", "ok", "okay", "works", "worked", "product",
    "item", "purchase", "stars", "star", "rating", "recommend",
    "recommended", "happy", "favorite", "best", "bad", "terrible",
    "disappointed", "beautiful", "wonderful"
}

GENERIC_PHRASES = {
    "great product", "good product", "excellent product", "love it",
    "love this", "works great", "five stars", "four stars", "three stars",
    "very good", "highly recommend", "great purchase"
}


def informative_tokens(text):
    toks = normalize_query(text).split()

    return [
        t for t in toks
        if (
            t not in STOPWORDS
            and t not in NOISE
            and len(t) > 1
            and not t.isdigit()
        )
    ]


def review_summary_quality(summary, product_text):
    q = normalize_query(summary)

    if not q:
        return None

    toks = q.split()

    if not (2 <= len(toks) <= 8):
        return None

    if q in GENERIC_PHRASES:
        return None

    if re.search(r"\b[1-5]\s*(star|stars)\b", q):
        return None

    if "star" in toks or "stars" in toks:
        return None

    q_info = set(informative_tokens(q))

    if not q_info:
        return None

    p_info = set(informative_tokens(product_text))

    overlap = q_info & p_info

    if len(overlap) < 1:
        return None

    return (
        2.0 * len(overlap) / max(len(q_info), 1)
        + 0.1 * len(q_info)
    )


def title_to_query(title, brand=""):
    title_tokens = normalize_query(title).split()
    brand_tokens = set(normalize_query(brand).split())

    kept = []

    for token in title_tokens:
        if token in brand_tokens:
            continue

        if token in {
            "new", "pack", "set", "pcs", "piece", "pieces"
        }:
            continue

        kept.append(token)

    if len(kept) > 8:
        kept = kept[:8]

    if len(kept) < 2:
        kept = title_tokens[:8]

    return normalize_query(" ".join(kept))


# ============================================================
# 8. BUILD SOURCE QUERY POOL
# ============================================================

# Prefer products that have Amazon relation links.
source_priority = products_full[
    products_full["direct_relation_count"] > 0
].copy()

print("\nProducts with at least one direct Amazon relation:", len(source_priority))

# Map product text
product_text_map = dict(
    zip(
        products_full["product_id"],
        products_full["product_text"]
    )
)

# Filter reviews to source-priority ASINs
priority_asins = set(source_priority["product_id"])

reviews_for_queries = reviews[
    reviews["asin"].isin(priority_asins)
].copy()

summary_map = {}

if "summary" in reviews_for_queries.columns:
    scored = []

    for _, r in tqdm(
        reviews_for_queries[["asin", "summary"]].iterrows(),
        total=len(reviews_for_queries),
        desc="Filtering review summaries"
    ):
        asin = r["asin"]

        score = review_summary_quality(
            r["summary"],
            product_text_map.get(asin, "")
        )

        if score is not None:
            scored.append({
                "asin": asin,
                "query": normalize_query(r["summary"]),
                "score": score
            })

    if scored:
        scored_df = pd.DataFrame(scored)

        scored_df = (
            scored_df
            .sort_values("score", ascending=False)
            .drop_duplicates("asin", keep="first")
        )

        summary_map = dict(
            zip(scored_df["asin"], scored_df["query"])
        )


source_rows = []

for _, row in source_priority.iterrows():
    pid = row["product_id"]

    if pid in summary_map:
        q = summary_map[pid]
        source = "review_summary_filtered"
    else:
        q = title_to_query(
            row["title"],
            row["brand"]
        )
        source = "product_title"

    q = normalize_query(q)

    if 2 <= len(q.split()) <= 8:
        source_rows.append({
            "source_product_id": pid,
            "query": q,
            "query_source": source,
            "direct_relation_count": int(row["direct_relation_count"])
        })

queries = pd.DataFrame(source_rows)

if len(queries) == 0:
    raise RuntimeError(
        "No source queries were created from products with relation links."
    )

queries = queries.drop_duplicates("query").reset_index(drop=True)

# Prefer sources with more relation evidence.
queries = queries.sort_values(
    "direct_relation_count",
    ascending=False
).reset_index(drop=True)

if len(queries) > N_QUERIES:
    # random sample from the top-quality relation-backed pool
    pool_n = min(len(queries), max(N_QUERIES * 3, N_QUERIES))
    queries = queries.head(pool_n).sample(
        N_QUERIES,
        random_state=RANDOM_STATE
    ).reset_index(drop=True)

queries.insert(
    0,
    "query_id",
    [f"q_{i:05d}" for i in range(len(queries))]
)

print("\nQueries built:", queries.shape)
display(queries["query_source"].value_counts().rename_axis("query_source").reset_index(name="count"))
display(queries.head(20))


# ============================================================
# 9. BUILD A 10K CATALOG THAT PRESERVES RELATIONS
# ============================================================

full_by_id = products_full.set_index("product_id", drop=False)

source_ids = set(queries["source_product_id"])

# Include source products.
catalog_ids = set(source_ids)

# Include up to several direct related products for each source.
for pid in source_ids:
    rels = full_by_id.loc[pid, "direct_related"]

    for rid in rels[:5]:
        catalog_ids.add(rid)

# Add one level of second-hop products for better label-1 coverage.
first_hop_ids = list(catalog_ids - source_ids)

for rid in first_hop_ids:
    if rid in full_by_id.index:
        second = full_by_id.loc[rid, "direct_related"]

        for sid in second[:2]:
            catalog_ids.add(sid)

# Fill remaining catalog randomly.
if len(catalog_ids) < N_PRODUCTS:
    remaining_ids = list(full_ids - catalog_ids)

    add_n = min(
        N_PRODUCTS - len(catalog_ids),
        len(remaining_ids)
    )

    if add_n > 0:
        rng = np.random.default_rng(RANDOM_STATE)
        catalog_ids.update(
            rng.choice(
                remaining_ids,
                size=add_n,
                replace=False
            ).tolist()
        )

# If relation expansion somehow made it too large, preserve all source IDs
# plus their first two direct relations, then fill up to N_PRODUCTS.
if len(catalog_ids) > N_PRODUCTS:
    must_keep = set(source_ids)

    for pid in source_ids:
        rels = full_by_id.loc[pid, "direct_related"]
        must_keep.update(rels[:2])

    optional = list(catalog_ids - must_keep)

    need = max(
        0,
        N_PRODUCTS - len(must_keep)
    )

    rng = np.random.default_rng(RANDOM_STATE)

    if need > 0 and optional:
        chosen = rng.choice(
            optional,
            size=min(need, len(optional)),
            replace=False
        ).tolist()
    else:
        chosen = []

    catalog_ids = must_keep | set(chosen)

products = products_full[
    products_full["product_id"].isin(catalog_ids)
].copy().reset_index(drop=True)

print("\nFinal relation-preserving catalog:", products.shape)

catalog_id_set = set(products["product_id"])

# Recompute relation lists inside final catalog.
products["also_buy_in_sample"] = products["also_buy_in_catalog"].apply(
    lambda xs: [x for x in xs if x in catalog_id_set]
)

products["also_view_in_sample"] = products["also_view_in_catalog"].apply(
    lambda xs: [x for x in xs if x in catalog_id_set]
)

products["direct_related_in_sample"] = products.apply(
    lambda r: list(dict.fromkeys(
        r["also_buy_in_sample"] + r["also_view_in_sample"]
    )),
    axis=1
)

products["direct_relation_count_in_sample"] = products[
    "direct_related_in_sample"
].apply(len)

sample_by_id = products.set_index("product_id", drop=False)

source_relation_check = pd.DataFrame({
    "source_product_id": queries["source_product_id"]
})

source_relation_check["direct_relations_in_sample"] = (
    source_relation_check["source_product_id"]
    .map(
        products.set_index("product_id")[
            "direct_relation_count_in_sample"
        ]
    )
    .fillna(0)
    .astype(int)
)

print("\nSource products with relation candidates preserved:")
display(
    source_relation_check[
        "direct_relations_in_sample"
    ].describe().to_frame("value")
)


# ============================================================
# 10. BUILD RELATION GRAPH FOR 2-HOP WEAK POSITIVES
# ============================================================

graph = {
    pid: rels
    for pid, rels in zip(
        products["product_id"],
        products["direct_related_in_sample"]
    )
}

brand_to_ids = defaultdict(list)

for _, row in products.iterrows():
    brand = clean_text(row["brand"]).lower().strip()

    if brand:
        brand_to_ids[brand].append(row["product_id"])


def second_hop_candidates(source_pid):
    direct = set(graph.get(source_pid, []))
    second = []

    for rid in direct:
        for sid in graph.get(rid, []):
            if sid != source_pid and sid not in direct:
                second.append(sid)

    return list(dict.fromkeys(second))


# ============================================================
# 11. TF-IDF FOR HARD-NEGATIVE MINING ONLY
# ============================================================
#
# IMPORTANT:
# TF-IDF is NOT used to define positive relevance labels in V4.
# It is only used to find lexically confusing NEGATIVES.
# ============================================================

vectorizer = TfidfVectorizer(
    lowercase=True,
    stop_words="english",
    max_features=40000,
    ngram_range=(1, 2),
    min_df=1
)

product_matrix = vectorizer.fit_transform(
    products["product_text"]
)

query_matrix = vectorizer.transform(
    queries["query"]
)

pid_to_idx = {
    pid: i
    for i, pid in enumerate(products["product_id"])
}

all_indices = np.arange(len(products))


# ============================================================
# 12. BUILD V4 GRADED RELEVANCE DATA
# ============================================================

pair_rows = []
skipped_queries = []

for qi, qrow in tqdm(
    queries.iterrows(),
    total=len(queries),
    desc="Building V4 relevance labels"
):
    qid = qrow["query_id"]
    query = qrow["query"]
    source_pid = qrow["source_product_id"]

    if source_pid not in pid_to_idx:
        skipped_queries.append((qid, "source_missing"))
        continue

    source_idx = pid_to_idx[source_pid]
    source_row = products.iloc[source_idx]

    # ------------------------------
    # Label 3: source product
    # ------------------------------
    label3 = [source_pid]

    # ------------------------------
    # Label 2: direct Amazon relations
    # Prefer also_buy, then also_view.
    # ------------------------------
    buy_rel = list(source_row["also_buy_in_sample"])
    view_rel = [
        x for x in source_row["also_view_in_sample"]
        if x not in buy_rel
    ]

    direct_ordered = buy_rel + view_rel

    label2 = direct_ordered[:N_LABEL2]

    # If only one direct relation exists, that's okay.
    # We do NOT create fake positive relations from TF-IDF.

    # ------------------------------
    # Label 1: second-hop relation
    # ------------------------------
    used_ids = set(label3 + label2)

    second = [
        x for x in second_hop_candidates(source_pid)
        if x in catalog_id_set and x not in used_ids
    ]

    label1 = second[:N_LABEL1]
    used_ids.update(label1)

    # Same-brand weak fallback, if needed.
    if len(label1) < N_LABEL1:
        brand_key = clean_text(source_row["brand"]).lower().strip()

        if brand_key:
            same_brand = [
                x for x in brand_to_ids.get(brand_key, [])
                if x not in used_ids
            ]

            need = N_LABEL1 - len(label1)
            extra = same_brand[:need]
            label1.extend(extra)
            used_ids.update(extra)

    # ------------------------------
    # Label 0: hard negatives
    # ------------------------------
    sims = linear_kernel(
        query_matrix[qi],
        product_matrix
    ).ravel()

    ranked_idx = np.argsort(-sims)

    known_related = set(label3 + label2 + label1)

    # Exclude ALL direct and second-hop graph relations from negatives.
    known_related.update(
        graph.get(source_pid, [])
    )
    known_related.update(
        second_hop_candidates(source_pid)
    )

    hard_neg_ids = []

    for idx in ranked_idx:
        pid = products.iloc[idx]["product_id"]

        if pid in known_related:
            continue

        hard_neg_ids.append(pid)

        if len(hard_neg_ids) >= N_HARD_NEG:
            break

    used_ids.update(hard_neg_ids)

    # Random negative from unrelated pool.
    random_pool = [
        pid
        for pid in products["product_id"]
        if pid not in known_related
        and pid not in used_ids
    ]

    if random_pool:
        random_neg_ids = [
            random.choice(random_pool)
        ]
    else:
        random_neg_ids = []

    # ------------------------------
    # Materialize candidates
    # ------------------------------
    candidate_specs = []

    candidate_specs += [
        (pid, 3, "source_product")
        for pid in label3
    ]

    for pid in label2:
        if pid in buy_rel:
            relation = "direct_also_buy"
        else:
            relation = "direct_also_view"

        candidate_specs.append(
            (pid, 2, relation)
        )

    candidate_specs += [
        (pid, 1, "second_hop_or_same_brand")
        for pid in label1
    ]

    candidate_specs += [
        (pid, 0, "lexical_hard_negative")
        for pid in hard_neg_ids
    ]

    candidate_specs += [
        (pid, 0, "random_negative")
        for pid in random_neg_ids
    ]

    for pid, label, relation in candidate_specs:
        if pid not in sample_by_id.index:
            continue

        p = sample_by_id.loc[pid]

        idx = pid_to_idx[pid]

        pair_rows.append({
            "query_id": qid,
            "query": query,
            "product_id": pid,
            "title": p["title"],
            "description": p["description"],
            "image_url": p["image_url"],
            "category": p["category"],
            "brand": p["brand"],
            "price": p["price"],
            "average_rating": p["average_rating"],
            "review_count": p["review_count"],
            "price_missing": p["price_missing"],
            "description_missing": p["description_missing"],
            "image_missing": p["image_missing"],
            "brand_missing": p["brand_missing"],
            "relevance_label": int(label),
            "label_relation": relation,
            "label_source": "week1_v4_amazon_relation_graph",
            # Diagnostic only; not used for positive-label construction.
            "tfidf_similarity_diagnostic": float(sims[idx])
        })

ranking_pairs = pd.DataFrame(pair_rows)

ranking_pairs = ranking_pairs.drop_duplicates(
    ["query_id", "product_id"]
).reset_index(drop=True)

print("\nRanking pairs:", ranking_pairs.shape)


# ============================================================
# 13. QUALITY CHECKS
# ============================================================

display(Markdown("## V4 Label Distribution"))

label_dist = (
    ranking_pairs["relevance_label"]
    .value_counts()
    .sort_index()
    .rename_axis("relevance_label")
    .reset_index(name="count")
)

label_dist["rate"] = (
    label_dist["count"]
    / len(ranking_pairs)
)

display(label_dist)


display(Markdown("## V4 Label Relation Distribution"))

relation_dist = (
    ranking_pairs["label_relation"]
    .value_counts()
    .rename_axis("label_relation")
    .reset_index(name="count")
)

display(relation_dist)


# How many queries actually have graph positives?
per_query = (
    ranking_pairs.groupby("query_id")
    .agg(
        candidate_count=("product_id", "count"),
        label3_count=("relevance_label", lambda s: int((s == 3).sum())),
        label2_count=("relevance_label", lambda s: int((s == 2).sum())),
        label1_count=("relevance_label", lambda s: int((s == 1).sum())),
        label0_count=("relevance_label", lambda s: int((s == 0).sum()))
    )
    .reset_index()
)

display(Markdown("## Candidates / Positive Coverage per Query"))
display(per_query.describe())


# ============================================================
# 14. SANITY CHECK
# ============================================================

sample_qids = (
    ranking_pairs["query_id"]
    .drop_duplicates()
    .sample(
        n=min(10, ranking_pairs["query_id"].nunique()),
        random_state=RANDOM_STATE
    )
    .tolist()
)

sanity = (
    ranking_pairs[
        ranking_pairs["query_id"].isin(sample_qids)
    ]
    .sort_values(
        ["query_id", "relevance_label"],
        ascending=[True, False]
    )[
        [
            "query_id",
            "query",
            "relevance_label",
            "label_relation",
            "title",
            "brand",
            "category",
            "tfidf_similarity_diagnostic"
        ]
    ]
)

display(Markdown("## V4 Sanity Check — 10 Random Queries"))
display(sanity)


# ============================================================
# 15. DATA QUALITY / MISSINGNESS
# ============================================================

missing_rows = []

for col in [
    "title", "description", "image_url", "brand",
    "price", "average_rating", "review_count"
]:
    if col in ["title", "description", "image_url", "brand"]:
        miss = (
            products[col]
            .fillna("")
            .astype(str)
            .str.strip()
            .eq("")
        )
    else:
        miss = products[col].isna()

    missing_rows.append({
        "field": col,
        "missing_count": int(miss.sum()),
        "missing_rate": float(miss.mean())
    })

missing_report = pd.DataFrame(
    missing_rows
).sort_values(
    "missing_rate",
    ascending=False
).reset_index(drop=True)

display(Markdown("## Missing Values"))
display(missing_report)


# ============================================================
# 16. SAVE OUTPUTS
# ============================================================

products.to_parquet(
    OUT_DIR / "cleaned_products_10k.parquet",
    index=False
)

queries.to_parquet(
    OUT_DIR / "normalized_queries.parquet",
    index=False
)

ranking_pairs.to_parquet(
    OUT_DIR / "query_product_relevance.parquet",
    index=False
)

# CSV versions
products.head(1000).to_csv(
    OUT_DIR / "cleaned_products_sample.csv",
    index=False
)

queries.to_csv(
    OUT_DIR / "normalized_queries.csv",
    index=False
)

ranking_pairs.to_csv(
    OUT_DIR / "query_product_relevance.csv",
    index=False
)

label_dist.to_csv(
    OUT_DIR / "relevance_label_distribution.csv",
    index=False
)

relation_dist.to_csv(
    OUT_DIR / "label_relation_distribution.csv",
    index=False
)

per_query.to_csv(
    OUT_DIR / "query_candidate_coverage.csv",
    index=False
)

missing_report.to_csv(
    OUT_DIR / "missing_value_report.csv",
    index=False
)

sanity.to_csv(
    OUT_DIR / "sanity_check_10_queries.csv",
    index=False
)

relation_summary.to_csv(
    OUT_DIR / "amazon_relation_graph_summary.csv",
    index=False
)


# ============================================================
# 17. README
# ============================================================

readme = f"""
AMAZON REVIEW DATA 2018 — WEEK 1 PREPROCESSING V4
=================================================

Dataset:
{DATASET_CATEGORY}

Products in development catalog:
{len(products):,}

Pseudo queries:
{len(queries):,}

Query-product pairs:
{len(ranking_pairs):,}


WHY V4
------
Earlier versions used category + lexical similarity to create
positive relevance labels. That created weak/circular labels.

V4 instead uses Amazon-provided product relationship fields:
- also_buy
- also_view / also_viewed

TF-IDF is NOT used to define positive labels in V4.
It is used only to mine difficult negative examples.


RELEVANCE LABELS
----------------
3 = source product associated with the pseudo query

2 = DIRECT Amazon relation
    - also_buy
    - also_view / also_viewed

1 = weaker product relation
    - second-hop relation in Amazon graph
    - same-brand fallback when necessary

0 = unrelated candidate
    - lexical hard negative
    - random negative


IMPORTANT LIMITATION
--------------------
This is still NOT true Amazon search relevance ground truth.

Amazon Review Data 2018 does not provide:
- real search queries
- search click logs
- explicit query-product relevance judgments

Therefore:
- queries are pseudo queries
- relevance labels are heuristic proxies

V4 is better suited for baseline ranking experiments because
positive labels come from Amazon's product relationship graph
rather than from the same lexical similarity features that will
be evaluated later.


WEEK-2 USE
----------
Use:
- normalized_queries.parquet
- query_product_relevance.parquet
- cleaned_products_10k.parquet

for:
- BM25
- TF-IDF baseline
- pointwise XGBoost
- NDCG@10 / MAP@10 / MRR evaluation
"""

with open(
    OUT_DIR / "README_DATA_V4.txt",
    "w",
    encoding="utf-8"
) as f:
    f.write(readme.strip())


# ============================================================
# 18. ZIP + AUTO DOWNLOAD
# ============================================================

zip_file = shutil.make_archive(
    "/content/amazon_week1_v4_processed_data",
    "zip",
    root_dir=OUT_DIR
)

print("\n==========================================")
print("V4 PREPROCESSING COMPLETE")
print("ZIP:", zip_file)
print("==========================================")

from google.colab import files
files.download(zip_file)
